# 第9周 · 周六：工具变量（IV）与断点回归（RDD）——当自然界的"随机"来帮忙\n\n> 难度：⭐⭐⭐⭐⭐ | 定位：理解IV和RDD的识别策略，能判断什么时候用哪个、用了意味着什么\n> 前置依赖：周一至周五的因果推断框架 | 后续用途：W10-W12综合案例中的进阶因果推断\n> 面试地位：高级数据分析/经济学岗位必问，展示你因果推断深度的关键\n\n---\n\n## 📘 区块1：核心概念与第一性原理\n\n### 1.1 IV和RDD的"自然实验"哲学\n\n如果说AB实验是"人为创造的随机"，那么IV和RDD就是在寻找"自然界中的随机"：\n\n- **IV**：寻找一个"只通过处理变量影响结果"的外部冲击。如：抽签（影响是否参军）、距离（影响是否就诊）、天气（影响是否出行）。\n- **RDD**：利用一个"断点阈值"制造的局部随机。如：分数线（上线/不上线）、年龄线（满18岁/没满）、地理边界（边界两侧）。\n\n**核心区别**：IV需要找一个"工具变量"（通常很难！），RDD只需要确认"处理分配是基于一个断点"（通常更容易验证）。\n\n### 1.2 工具变量(IV)的识别逻辑\n\n当处理变量T和未观测混淆U相关时，OLS有偏。IV引入一个工具变量Z，满足两个条件：\n\n**条件1：相关性（Relevance）**——Z必须与T相关。\n$Cov(Z, T) \\neq 0$（可检验！）\n\n**条件2：外生性（Exogeneity）**——Z只能通过T影响Y。\n$Cov(Z, \\varepsilon) = 0$（不可检验！！）\n\n**条件3：排他性（Exclusion Restriction）**——Z对Y没有直接影响。\nZ→T→Y是唯一的因果路径。\n\n**IV估计量（Wald Estimator）**：\n$$\\beta_{IV} = \\frac{E[Y|Z=1] - E[Y|Z=0]}{E[T|Z=1] - E[T|Z=0]} = \\frac{\\text{简化式效应}}{\\text{第一阶段效应}}$$\n\n用白话：分子是"抽到签的人vs没抽到签的人"的收入差，分母是"抽到签vs没抽到签时参军概率的差"。两者的比值就是参军的因果效应。\n\n**关键洞察**：IV不估计ATE，它估计的是LATE（局部平均处理效应）——只对那些"因Z而改变处理状态的人"有效。在参军例子中，LATE针对的是"抽到签就去参军，没抽到就不去"的人（遵从者，Compliers）。对那些"不管怎样都会参军"（Always-takers）或"不管怎样都不参军"（Never-takers）的人，IV无法估计效应。\n\n### 1.3 断点回归(RDD)的直觉\n\nRDD利用处理分配中的"跳跃"（Jump / Discontinuity）来识别因果效应。\n\n**RDD的识别策略**：在断点（cutoff）附近，处于断点两侧的人是"几乎一样"的——除了刚好过线的人被处理了，刚好没过线的人没被处理。这创造了一个"局部的随机化"。\n\n**两种RDD**：\n- **Sharp RDD**：处理在断点处从0跳到1（确定性）。如：≥600分一定录取，<600分一定不录取。\n- **Fuzzy RDD**：处理概率在断点处跳跃，但不是从0到1（概率性）。如：≥600分录取概率大幅提升但不是100%。→Fuzzy RDD本质上是IV（用"是否过线"作为"是否录取"的IV）。\n\n**RDD的关键假设**：除了处理分配本身，其他所有因素在断点处都是连续的。\n如果这个假设成立→断点处的Y跳跃 = 处理效应。\n\n### 1.4 面试高频考点\n\n**面试官常问**：\n- "IV的两个条件是什么？哪个可检验、哪个不能？"\n- "什么是LATE？为什么IV估计的是LATE而不是ATE？"\n- "RDD和IV有什么关系？"\n- "你工作中用过IV或RDD吗？如果用过，工具变量是什么？"\n\n**回答要点**：\n- 相关性(可检验)+外生性(不可检验——需要理论论证)\n- LATE=只对Compliers的效应；IV不能估计Always-takers和Never-takers的效应\n- Fuzzy RDD = IV（用断点作为IV），Sharp RDD = 在断点处直接比较\n- 工具变量的选择论证比技术实现更重要——好的IV是"故事"而不是"代码"

## 🧩 区块2：知识点详细拆解\n\n### 2.1 2SLS（两阶段最小二乘）——IV的标准实现\n\n**为什么重要**：2SLS是IV估计的标准方法，理解它的两个阶段有助于理解IV在做什么。\n\n**第一阶段**：用工具变量Z和协变量X预测处理变量T\n```\nT = γ₀ + γ₁·Z + γ₂·X + ν\n→ 得到T的预测值 T̂\n```\n\n**第二阶段**：用T̂替代T来估计结果方程\n```\nY = β₀ + β₁·T̂ + β₂·X + ε\n→ β₁就是IV估计量\n```\n\n**第一阶段F统计量**：\n- F > 10：强工具变量 ✓（Stock & Yogo, 2005的经验准则）\n- F < 10：弱工具变量 ⚠（即使相关性显著，IV估计也可能有严重偏误）\n- 如果F < 10→不要用2SLS，考虑弱工具变量稳健方法（LIML/Anderson-Rubin检验）\n\n**为什么F>10很重要**：弱工具变量的2SLS估计不仅方差大，而且是有偏的（即使在无限样本下也向OLS偏误方向偏）。F<10时，2SLS的偏误可以是OLS偏误的10-20%。\n\n### 2.2 工具变量的选择——最难的步骤\n\n**为什么重要**：找到好的IV是整个因果推断中最难的事。在互联网行业，好的IV更是稀缺。\n\n**经典IV示例（来自已发表论文）**：\n\n| 研究问题 | 处理变量 | 工具变量 | 为什么有效 |\n|---------|---------|---------|------------|\n| 服兵役对收入的影响 | 是否服兵役 | 抽签号码 | 抽签→服兵役(相关性)，抽签随机(外生性) |\n| 教育对收入的影响 | 受教育年限 | 出生季度+义务教育法 | 出生季度→入学年龄→教育年限 |\n| 电视对考试成绩的影响 | 看电视时间 | 是否有有线电视 | 有线电视接入→看电视时长 |\n| 医院对健康的影响 | 去哪个医院 | 距离最近医院 | 距离→医院选择(相关性)，距离与健康无关(外生性?) |\n| 移民对工资的影响 | 移民数量 | 历史上的移民网络 | 历史移民→现在移民(相关性)，历史与现在工资无关(外生性?) |\n\n**在互联网行业的IV可能性**：\n- 服务器延迟→用户看到的内容→用户行为\n- 网络中断→广告曝光→购买\n- A/B测试中的"non-compliance"（用户没按分组做）→实际处理\n- **Encouragement Design**：随机给某些用户"推送通知推荐使用某功能"（随机IV！），用"是否收到推送"作为"是否使用功能"的IV\n\n### 2.3 RDD的带宽选择——"多宽算阈值附近"\n\n**为什么重要**：带宽是RDD中最关键的建模选择——太宽（偏误大）、太窄（方差大）。\n\n**方法1：交叉验证（IK法, Imbens & Kalyanaraman 2012）**\n- 目标：最小化MSE(偏误²+方差)\n- 在数据驱动的意义上"最优"\n- 许多RDD论文的首选方法\n\n**方法2：CCT法（Calonico, Cattaneo & Titiunik 2014）**\n- 在IK法基础上修正了偏误\n- 提供稳健的置信区间\n\n**方法3：主观选择+敏感性分析**\n- 用0.5×IK, 1×IK, 2×IK三种带宽展示结果\n- 如果三种带宽结论一致→增强可信度\n- 如果结论对带宽敏感→不稳健\n\n### 2.4 RDD的操纵检验(McCrary Test)\n\n**为什么重要**：RDD的致命威胁是"个体可以操纵forcing variable来到断点某侧"。如学生知道600分是分数线→拼命考601分→造成断点附近左侧人少、右侧人多→操纵成立→RDD不可信。\n\n**McCrary检验**：检验forcing variable的密度函数在断点处是否连续。\n- 如果密度在断点处跳跃→有人为操纵→RDD假设违反\n- 如果密度连续→无人为操纵→通过检验 ✓\n\n### 2.5 RDD的安慰剂检验——用"假断点"验证\n\n**为什么重要**：和DID一样，RDD也需要安慰剂检验来排除"跳跃是断点本身造成的"的可能性。\n\n**做法**：\n1. 在forcing variable的其他位置（不是真正的断点）做RDD\n2. 这些假断点处的估计效应应该接近0（不显著）\n3. 如果某个假断点也"显著"→说明断点处有自然的非线性→RDD结果不可信\n\n### 2.6 RDD vs IV vs DID vs PSM——四种方法的全集\n\n**为什么重要**：这是W9的"总复习"——四种方法各有什么优劣。\n\n| 维度 | DID | PSM | IV | RDD |\n|------|-----|-----|-----|-----|\n| 识别来源 | 时间维度的趋势差 | 横截面的相似性 | 工具变量的外生冲击 | 断点附近的局部随机 |\n| 核心假设 | 平行趋势 | 可忽略性(无遗漏混淆) | 外生性+排他性 | 断点处连续性 |\n| 核心假设可检验？ | 部分(处理前趋势) | 部分(平衡性) | 相关性可检验，外生性不可 | 部分(McCrary检验) |\n| 估计的目标 | ATT(通常) | ATT | LATE(Compliers) | LATE(断点附近) |\n| 对未观测混淆的处理 | 控制不随时间变化的 | 假设不存在 | 用Z的外生性"隔离" | 假设断点附近局部随机 |\n| 互联网行业可用性 | 高 | 高 | 低(好IV稀缺) | 中(需要明确断点) |\n\n### 2.7 IV的常见误用——"用工具变量来钓鱼"\n\n**为什么重要**：IV是因果推断中"最容易被滥用"的方法——因为找到一个"看起来像IV"的变量太容易了。\n\n**误用1："只用相关性判断IV"**\n- "这个变量和处理相关，所以它是好的IV"——完全忽略了外生性条件！\n- 实际上：相关性只是"入场券"，外生性才是"通行证"\n\n**误用2："不做第一阶段F检验"**\n- 第一阶段看似显著（t>2）但F=3→弱工具变量→2SLS的偏误比OLS还大！\n\n**误用3："过度识别时追求J统计量不显著就好"**\n- J检验（Sargan/Hansen过度识别检验）的零假设是"所有IV都是外生的"\n- J检验不显著≠"外生性得证"→它只是说"没有证据反对外生性"（在多个IV时）\n- 如果一个IV是内生的，J检验（在统计上）可能仍然不显著——尤其是样本量不大的时候

## 💻 区块3：动手练习\n\n### 步骤0：环境准备

In [ ]:
import numpy as np; import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from scipy import stats
import warnings; warnings.filterwarnings('ignore')
np.random.seed(42)
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
print('='*60)
print('工具变量(IV)与断点回归(RDD)——动手练习')
print('='*60)

### 步骤1：工具变量——2SLS从理论到代码

1.1 模拟"教育对收入"的IV估计（工具变量=出生季度）
1.2 验证第一阶段F统计量

In [ ]:
# 步骤1.1：IV模拟——教育对收入的影响
print('【步骤1.1】IV估计：教育对收入的影响')
print('-'*50)

n = 1000
# 未观测的能力(混淆变量)
ability = np.random.normal(0, 1, n)

# 工具变量：出生季度（影响受教育年限——义务教育法规定满6岁才能入学）
birth_quarter = np.random.choice([1, 2, 3, 4], n)
# 出生季度靠后的人→入学晚→可能少读一些书

# 第一阶段：教育年限 = f(出生季度, 能力)
quarter_effect = {-1: 0, 0: -0.3, 1: -0.6, 2: -0.5}  # Q1-Q4对教育的影响
education = 12 + quarter_effect[birth_quarter[0]] if len(set(birth_quarter))==1 else 12
education = np.array([12 + (-0.3*(q-1)) + 0.8*ability + np.random.normal(0, 1) for q in birth_quarter])

# 结果：收入 = f(教育, 能力)
true_edu_effect = 0.08  # 每多读一年书，收入增加8%
log_wage = 10 + true_edu_effect*education + 0.15*ability + np.random.normal(0, 0.2, n)

df_iv = pd.DataFrame({'edu': education, 'log_wage': log_wage, 
                      'Q2': (birth_quarter==2).astype(int),
                      'Q3': (birth_quarter==3).astype(int),
                      'Q4': (birth_quarter==4).astype(int),
                      'ability': ability})

# 朴素OLS（有偏——忽略了能力）
ols = LinearRegression().fit(df_iv[['edu']], df_iv['log_wage'])
print(f'OLS估计(有偏): {ols.coef_[0]:.4f} (真实={true_edu_effect})')
print(f'  → 高估了，因为能力和教育正相关')

# IV-2SLS
# 第一阶段：edu ~ Q2+Q3+Q4
first_stage = LinearRegression().fit(df_iv[['Q2','Q3','Q4']], df_iv['edu'])
df_iv['edu_hat'] = first_stage.predict(df_iv[['Q2','Q3','Q4']])

# 第一阶段F检验
ssr_restricted = np.sum((df_iv['edu'] - df_iv['edu'].mean())**2)
ssr_unrestricted = np.sum((df_iv['edu'] - df_iv['edu_hat'])**2)
n_restrictions = 3  # Q2, Q3, Q4三个变量
F_first = ((ssr_restricted - ssr_unrestricted) / n_restrictions) / (ssr_unrestricted / (n - 4))
print(f'\\n第一阶段F统计量: {F_first:.1f}')
print(f'{"✓ F>10, 强工具变量" if F_first>10 else "⚠ F<10, 弱工具变量!"} ')

# 第二阶段：log_wage ~ edu_hat
second_stage = LinearRegression().fit(df_iv[['edu_hat']], df_iv['log_wage'])
iv_estimate = second_stage.coef_[0]
print(f'\\nIV-2SLS估计: {iv_estimate:.4f} (真实={true_edu_effect})')
print(f'  → 比OLS接近真实值（但仍非完美——IV的方差大）')

In [ ]:
# 步骤1.2：可视化IV识别的逻辑
print('\n【步骤1.2】IV的识别逻辑——可视化')
print('-'*50)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 简化式(reduced form): Y ~ Z
for q, label, color in [(1,'Q1','blue'),(2,'Q2','green'),(3,'Q3','orange'),(4,'Q4','red')]:
    mask = birth_quarter == q
    axes[0].scatter(q, log_wage[mask].mean(), s=200, color=color, zorder=5)
axes[0].set_xlabel('出生季度'); axes[0].set_ylabel('平均log工资')
axes[0].set_title('简化式: Y~Z (只看Z和Y的关系)')

# 第一阶段: T ~ Z
for q, label, color in [(1,'Q1','blue'),(2,'Q2','green'),(3,'Q3','orange'),(4,'Q4','red')]:
    mask = birth_quarter == q
    axes[1].scatter(q, education[mask].mean(), s=200, color=color, zorder=5)
axes[1].set_xlabel('出生季度'); axes[1].set_ylabel('平均受教育年限')
axes[1].set_title('第一阶段: T~Z (Z影响T吗?)')

# IV = 简化式/第一阶段
axes[2].scatter(df_iv['edu_hat'], df_iv['log_wage'], alpha=0.3, s=10)
axes[2].plot(df_iv['edu_hat'], second_stage.predict(df_iv[['edu_hat']]), 'r-', linewidth=2)
axes[2].set_xlabel('预测的教育年限(T_hat)'); axes[2].set_ylabel('log工资(Y)')
axes[2].set_title(f'第二阶段: Y~T_hat (IV={iv_estimate:.4f})')
plt.tight_layout(); plt.show()

### 步骤2：断点回归——Sharp RDD完整实现

2.1 模拟"奖学金(GPA≥80)对毕业率"的RDD
2.2 带宽敏感性分析

In [ ]:
# 步骤2.1：Sharp RDD——奖学金对毕业率的影响
print('\n【步骤2.1】RDD: 奖学金对毕业率的影响')
print('-'*50)

np.random.seed(2024)
n_rdd = 500

# forcing variable: GPA（0-100分）
gpa = np.random.uniform(50, 100, n_rdd)
# 处理：GPA≥80获得奖学金
scholarship = (gpa >= 80).astype(int)

# 结果：毕业率（%）——跟GPA连续相关+奖学金效应+噪音
true_scholar_effect = 5.0  # 奖学金提升毕业率5个百分点
graduation = (60 + 0.3*(gpa-70) + true_scholar_effect*scholarship 
              + np.random.normal(0, 3, n_rdd))

df_rdd = pd.DataFrame({'gpa': gpa, 'scholarship': scholarship, 'graduation': graduation})

print(f'断点: GPA=80')
print(f'获得奖学金: {scholarship.sum()}人')
print(f'真实奖学金效应: {true_scholar_effect}个百分点')

# RDD: 取断点附近的样本(带宽=5分)
for bw in [10, 5, 3]:
    near = (gpa >= 80-bw) & (gpa <= 80+bw)
    subset = df_rdd[near]
    treated_near = subset[subset.scholarship==1]['graduation'].mean()
    control_near = subset[subset.scholarship==0]['graduation'].mean()
    rdd_est = treated_near - control_near
    print(f'  带宽±{bw}: 处理组={treated_near:.1f}, 对照组={control_near:.1f}, RDD={rdd_est:.1f} (n={near.sum()})')

# 可视化
fig, ax = plt.subplots(figsize=(10, 6))
# 散点图（透明度调整）
ax.scatter(gpa[scholarship==0], graduation[scholarship==0], alpha=0.3, s=15, color='blue', label='无奖学金')
ax.scatter(gpa[scholarship==1], graduation[scholarship==1], alpha=0.3, s=15, color='red', label='有奖学金')

# 断点两侧分别拟合
for side, color in [('left', 'blue'), ('right', 'red')]:
    if side == 'left':
        mask = gpa < 80
    else:
        mask = gpa >= 80
    X_side = gpa[mask].reshape(-1, 1)
    y_side = graduation[mask]
    reg_side = LinearRegression().fit(X_side, y_side)
    x_plot = np.linspace(gpa[mask].min(), gpa[mask].max(), 50).reshape(-1, 1)
    ax.plot(x_plot, reg_side.predict(x_plot), color=color, linewidth=2.5)

ax.axvline(x=80, color='green', linestyle='--', linewidth=2, label='GPA=80(断点)')
ax.set_xlabel('GPA'); ax.set_ylabel('毕业率(%)')
ax.set_title(f'RDD: 奖学金对毕业率的因果效应\\n(断点处跳跃≈{rdd_est:.1f}个百分点)')
ax.legend()
plt.tight_layout(); plt.show()

print('\nRDD解读：')
print('  断点左侧(79分)和右侧(80分)的人几乎完全一样——除了是否获奖学金')
print('  断点处毕业率的跳跃 = 奖学金的因果效应')
print(f'  估计效应 ≈ {rdd_est:.1f}个百分点 (真实={true_scholar_effect})')

### 步骤3：挑战练习——McCrary检验和安慰剂检验

3.1 McCrary密度检验
3.2 RDD安慰剂检验

In [ ]:
# 步骤3.1：McCrary检验——检查forcing variable在断点处的密度是否连续
print('\n【步骤3.1】McCrary密度检验')
print('-'*50)

# 画GPA的直方图，检查断点附近是否有"堆积"或"缺口"
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 正常情况（我们模拟的数据，没有操纵）
axes[0].hist(gpa, bins=25, edgecolor='black', alpha=0.7)
axes[0].axvline(x=80, color='red', linestyle='--', linewidth=2)
axes[0].set_xlabel('GPA'); axes[0].set_title('McCrary检验: GPA分布\\n断点处无明显跳跃→无人为操纵 ✓')

# 模拟"有操纵"的情况（断点左侧少、右侧多）
gpa_manipulated = gpa.copy()
# 模拟"有人为操纵": GPA在78-80之间的人被"推"到了80以上
manip_mask = (gpa_manipulated >= 76) & (gpa_manipulated < 80)
gpa_manipulated[manip_mask] = gpa_manipulated[manip_mask] + np.random.uniform(2, 6, manip_mask.sum())

axes[1].hist(gpa_manipulated, bins=25, edgecolor='black', alpha=0.7)
axes[1].axvline(x=80, color='red', linestyle='--', linewidth=2)
axes[1].set_xlabel('GPA'); axes[1].set_title('McCrary检验: GPA分布(有操纵)\\n断点左侧少、右侧多→人为操纵 ⚠')
plt.tight_layout(); plt.show()

print('McCrary检验判断：')
left_density = ((gpa >= 75) & (gpa < 80)).sum() / 5
right_density = ((gpa >= 80) & (gpa <= 85)).sum() / 5
print(f'  [75,80)密度: {left_density:.0f}人/分')
print(f'  [80,85]密度: {right_density:.0f}人/分')
print(f'  密度比: {right_density/left_density:.2f} (应接近1.0)')
if 0.7 < right_density/left_density < 1.3:
    print('  → 密度比接近1 → 无人为操纵 ✓')
else:
    print('  → 密度比偏离1 → 可能有人为操纵 ⚠')

In [ ]:
# 步骤3.2：RDD安慰剂检验——在假断点处"不应有跳跃"
print('\n【步骤3.2】RDD安慰剂检验：假断点处的RDD应该≈0')
print('-'*50)

# 在真正的断点(80)之外的多个位置做RDD
fake_cutoffs = [60, 65, 70, 75, 85, 90, 95]
bw_placebo = 5

placebo_effects = []
print('假断点  | RDD估计 | 判断')
print('-'*40)
for cutoff in fake_cutoffs:
    near = (gpa >= cutoff-bw_placebo) & (gpa <= cutoff+bw_placebo)
    if near.sum() < 20:
        continue
    subset = df_rdd[near]
    treated = subset[subset.gpa >= cutoff]['graduation'].mean()
    control = subset[subset.gpa < cutoff]['graduation'].mean()
    effect = treated - control
    placebo_effects.append(effect)
    flag = '⚠ 显著!' if abs(effect) > 3 else '✓ 不显著'
    print(f'GPA={cutoff}  | {effect:+.2f}    | {flag}')

# 画图对比
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(range(len(placebo_effects)), placebo_effects, color=['green' if abs(e)<3 else 'red' for e in placebo_effects])
ax.axhline(y=0, color='black', linewidth=0.8)
ax.axhline(y=true_scholar_effect, color='blue', linestyle='--', label=f'真实断点效应={true_scholar_effect}')
ax.set_xticks(range(len(placebo_effects)))
ax.set_xticklabels([str(c) for c in fake_cutoffs[:len(placebo_effects)]])
ax.set_xlabel('假断点(GPA)'); ax.set_ylabel('RDD估计')
ax.set_title('RDD安慰剂检验：假断点不应有显著效应')
ax.legend()
plt.tight_layout(); plt.show()

print('\n结论：假断点的效应都接近0→真正的断点(80)处的跳跃确实来自奖学金→RDD可信 ✓')

## 📊 区块4：业务场景应用

### 场景1：促销短信的效果——一个"Encouragement Design"的IV

**背景**：电商平台想知道"使用新支付方式"对"客单价"的影响。但选择新支付方式的用户可能本身就是"愿意尝试新东西"的技术爱好者——他们的客单价可能本就不同。\n
**IV设计（Encouragement Design）**：\n
1. 随机选取一批用户，给他们推送"使用新支付方式享立减5元"的通知\n2. 收到通知的人中，有些因此使用了新支付方式（Compliers），有些本来就会用（Always-takers），有些始终不用（Never-takers）\n3. 工具变量Z = 是否被随机推送通知\n   - 相关性 ✓：推送→使用概率增加\n   - 外生性 ✓：推送是随机的，与用户特征无关\n4. IV估计 = (推送组客单价 - 未推送组客单价) / (推送组使用率 - 未推送组使用率)\n
**这就是A/B测试中的"non-compliance"场景——用户没按分组做。用IV来处理：**\n- ITT(Intent-to-Treat) = 直接比较分组（保守，低估真实效应）\n- IV = ITT / compliance rate（在compliance<100%时更接近真实效应）\n- LATE = 对"因推送而使用"的那部分人的效应\n
### 场景2：补贴券的RDD评估——"满300减50"的效果\n
**背景**：平台有个活动——购物满300元自动发一张50元优惠券。你想知道这张券对用户后续回购有没有效果。\n
**RDD设计**：\n\n1. Forcing variable：当次购物金额\n2. Cutoff：300元\n3. 处理：获得50元券（≥300元）vs 没获得（<300元）\n4. 结果：接下来7天内是否回购\n\n5. **为什么RDD有效**：花了298元和花了302元的人在消费能力/意愿上几乎没有差别——唯一的差别是302元的人拿到了券。\n\n6. **RDD估计**：取[270, 330]区间的用户→对比刚好≥300和刚好<300的两组人的回购率→约3%的回购提升。\n\n7. **注意事项**：\n   - 如果用户知道"满300发券"的规则，他们可能在订单上凑单→forcing variable有操纵→McCrary检验可能失败→需要谨慎\n   - 带宽选择：用IK最优带宽+敏感性分析(CCT)+手动选择三个带宽展示\n
### 场景3：面试中如何讲"我用过IV/RDD"\n
**面试官**："你工作中用过IV或RDD吗？"\n
**如果你有经验，回答框架**：\n1. **问题**："我们想知道XX功能对用户YY行为的影响，但存在选择偏差——用了功能的用户本来就更活跃。"\n2. **工具变量/断点**："我们利用了[某个自然实验/规则断点]——[描述Z或cutoff]。"\n3. **为什么有效**："Z和处理相关（展示F检验），Z的外生性由[理论/设计]保证（如随机推送/规则阈值）。"\n4. **结论+局限**："IV估计效应为X，但需要说明这估计的是LATE——只对[Compliers/断点附近人群]有效。"\n
**如果你没直接经验，诚实但展示理解**：\n"我在工作中还没有直接使用IV或RDD的机会——因为这需要特殊的工具变量或断点。但我理解它们的使用场景。比如，我们之前做的AB实验中有一个'non-compliance'问题（用户没按分组使用功能），那其实就可以用'随机分组'作为IV来估计'实际使用功能'的真实效果。如果将来遇到合适的断点场景（如满减活动的门槛），我会考虑用RDD。"

## ✅ 区块5：分级检验清单

### 🟢 初级检验（必须能独立完成的）
- [ ] 能说出IV的两个条件（相关性/外生性）及哪个可检验
- [ ] 能写出IV的Wald估计量公式
- [ ] 能区分Sharp RDD和Fuzzy RDD
- [ ] 能解释RDD的"断点"是什么意思

**答案提示**：
- 相关性可检验(第一阶段F>10)，外生性不可检验(需理论论证)
- β_IV = (E[Y|Z=1]-E[Y|Z=0]) / (E[T|Z=1]-E[T|Z=0])
- Sharp RDD: 处理在断点处0→1确定性跳跃；Fuzzy RDD: 概率跳跃(本质是IV)
- 断点=一个确定的阈值，用于分配处理

### 🟡 中级检验（能解释给同事听的原理）
- [ ] 能解释为什么第一阶段F<10是严重问题（弱工具变量的偏误和方差问题）
- [ ] 能解释LATE和ATE的区别
- [ ] 能说明RDD的带宽选择为什么重要
- [ ] 能说明McCrary检验在检查什么

**答案提示**：
- F<10→弱工具变量→2SLS不仅方差大,而且有偏(偏误朝OLS方向)，可能比OLS还差
- ATE针对所有人，LATE只针对Compliers(处理状态因Z而改变的人)
- 带宽太大→偏误大(断点远处的人不可比)，太小→方差大(样本少)
- McCrary检验检查forcing variable密度在断点处是否连续→检查是否有人为操纵

### 🔴 高级检验（能举一反三的迁移能力）
- [ ] 能评价一个IV分析的"工具变量选择"是否合理
- [ ] 能设计一个Encouragement Design来处理non-compliance
- [ ] 能判断什么场景下RDD比PSM/DID更合适
- [ ] 能解释IV的外生性假设为什么"无法从数据中证明"

**答案提示**：
- 好的IV需要"故事"——领域知识论证外生性，而非仅统计指标
- Encouragement Design = 随机推送+IV(随机分组作为使用行为的IV)
- 有明确断点(规则/制度)→RDD；无明确断点但有丰富协变量→PSM；有前后+对照→DID
- 外生性Cov(Z,ε)=0涉及不可观测的ε→无法用数据直接检验→需要理论/制度知识论证

## 🔗 区块6：跨模块连接

| 连接方向 | 关联内容 | 为什么有关 |
|---------|---------|----------|
| **← 前置依赖** | 因果推断基础(反事实/混淆/选择偏差) | IV和RDD是解决"未观测混淆"的两种"自然实验"策略 |
| **← 前置依赖** | 回归分析(OLS/2SLS) | IV的标准实现是2SLS |
| **← 前置依赖** | AB实验(W7-W8)的non-compliance | 实验中的non-compliance = IV的Encouragement Design |
| **→ 后续用途** | W10-W12综合案例 | 综合案例中展示"什么时候升级到IV/RDD" |
| **业务连接** | 所有"通过规则/制度"分配的场景 | 满减门槛/积分线/活跃度阈值→都可以考虑RDD |
| **业务连接** | AB实验中non-compliance问题的处理 | ITT vs IV vs Per-protocol |

### 横向连接：与其他分析方法的关联

**与AB实验（non-compliance）的连接**：
- AB实验中用户不遵从分组→ITT低估了真实处理效应→IV解决：用"随机分组"作为"实际处理"的IV
- 这可能是互联网数据分析师最常遇到IV的场景

**与机器学习的连接**：
- DeepIV：用深度神经网络同时建模第一阶段和第二阶段
- 因果森林+IV：非参数的第一阶段估计
\n**与RDD+DID的连接**：
- 如果RDD的断点是在某个时间点推出的（如"2024年1月1日起...")→结合RDD和DID→"断点DID"
- 可以在断点处用DID控制不随时间变化的混淆→增强可信度

## 📝 区块7：当日小结

> **IV和RDD是"自然实验"方法——当人工随机不可行时，利用自然界的"近似随机"来识别因果。IV需要一个同时满足"相关性"和"外生性"的工具变量——相关性可检验(F>10)，外生性不可检验(需理论论证)，这就是IV的阿喀琉斯之踵。IV估计的是LATE(局部平均处理效应)——只对"处理状态因Z而改变的人(Compliers)"有效，不是ATE。RDD利用断点附近的"局部随机"——断点两侧的人几乎一样，只是刚好过线的人被处理了。RDD比IV更容易验证(McCrary检验+安慰剂检验+带宽敏感性分析)，但只能估计断点附近的局部效应。两种方法都不常用——原因是好工具变量和好断点都极度稀缺。但在互联网行业中，Encouragement Design(随机推送→IV)和规则断点(满减门槛→RDD)是两个有潜力的应用方向。**

**三条金句带回家**：
1. IV估计的是LATE而不是ATE——它只能识别Compliers的效应，不能外推到Always-takers和Never-takers
2. 找到一个好工具变量的难度 >> 跑IV回归的难度——工具变量的选择是"故事"和"制度知识"，不是"代码"
3. RDD的黄金法则是"多看断点附近的散点图"——如果肉眼看不到跳跃，统计方法也看不到

### 🔖 课后拓展

1. **练习**：用RDD分析"积分满500分升金卡会员"对复购率的影响（模拟数据）

2. **思考**：在AB实验中，如果只有60%的用户遵从了分组（compliance rate=60%），ITT估计=+5%。用IV估计（用分组作为实际处理的IV），真实处理效应是多少？IV估计的有效人群是谁？

3. **复习**：明天是周日复习——回顾W9所有的因果推断方法，画出选择决策树

4. **面试准备**：
   - "IV的两个条件是什么？哪个可检验？"
   - "什么是LATE？为什么IV只能估计LATE？"
   - "你见过的好的工具变量有哪些？为什么它们'好'？"

---

*03analyst_thinking · Week 9 Day 6 · 2026-05-21*